Imports

In [ ]:
# Imports
import os
import json
from pathlib import Path
import csv
from collections import Counter
from itertools import product
from datetime import datetime
from IPython.display import display
from tqdm.auto import tqdm
import time
import threading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, precision_recall_curve)
import xgboost as xgb
from imblearn.over_sampling import SMOTE

from utils import load_and_extract_entities

Config

In [ ]:
# Data and horizon (must match pre-processing output)
DATA = "nph"   # 'pituitary' | 'nph'
HORIZON = "firstcontact"   # None (baseline) | 30 | 90 | 365 | "firstcontact"

# Model and sampling
ALGORITHM = "logistic_regression"    # 'random_forest' | 'logistic_regression' | 'xgboost' | 'svm' | 'knn'
SAMPLING = "none"    # 'none' | 'class_weights' | 'smote'

# CV and randomness
CV_FOLDS = 5
RANDOM_STATE = 42
SCORING_METRIC = "roc_auc"

print("DATA:", DATA, "| HORIZON:", HORIZON, "| ALGORITHM:", ALGORITHM, "| SAMPLING:", SAMPLING)

In [ ]:
# --- Derived paths (do not edit) ---
_prefix = "pa" if DATA == "pituitary" else "nph"
_base = Path("timelines")
if HORIZON is None:
    TREATMENT_DIR = _base / f"{_prefix}_treatment_timelines_clean"
    CONTROL_DIR = _base / f"{_prefix}_control_timelines_clean"
else:
    TREATMENT_DIR = _base / f"{_prefix}_treatment_timelines_clean_h{HORIZON}"
    CONTROL_DIR = _base / f"{_prefix}_control_timelines_clean_h{HORIZON}"

RESULTS_FILE = f"{_prefix}_bag_of_words_experiment_results.csv"

# --- Sanity print ---
print("TREATMENT_DIR:", TREATMENT_DIR, "exists:", TREATMENT_DIR.exists())
print("CONTROL_DIR  :", CONTROL_DIR, "exists:", CONTROL_DIR.exists())
print("RESULTS_FILE:", RESULTS_FILE)

Pre-Processing

In [ ]:
# Load treatment and control data
treatment_data = load_and_extract_entities(TREATMENT_DIR, 1)
control_data = load_and_extract_entities(CONTROL_DIR, 0)

# Combine into a single list
all_patient_data = treatment_data + control_data

print(f"\nLoaded {len(treatment_data)} treatment patients and {len(control_data)} control patients.")
print(f"Total patients: {len(all_patient_data)}")

In [ ]:
#Check sample
print(f"\nSample patient data:")
if all_patient_data:
    print(f"Patient {all_patient_data[0]['patient_id']}: {all_patient_data[0]['num_entities']} entities")
    print(f"First few entities: {all_patient_data[0]['entities'][:5] if all_patient_data[0]['entities'] else 'None'}")

In [ ]:
# Prepare DataFrame
agg_df = pd.DataFrame(all_patient_data)

# Shuffle the DataFrame to ensure patient order is random
agg_df = agg_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Display summary
print(f"Total patients: {len(agg_df)}")
print(f"\nLabel distribution:")
print(agg_df['label'].value_counts())
print(f"\nEntity count summary:")
print(agg_df['num_entities'].describe())
display(agg_df.head())

In [ ]:
# Data Splitting
train_df, test_df = train_test_split(
    agg_df,
    test_size=0.2,
    stratify=agg_df["label"],
    random_state=RANDOM_STATE
)

print("Data split into training and testing sets:")
print(f"Training set size: {len(train_df)}")
print(f"Test set size    : {len(test_df)}")

print("\nLabel distribution in training set:")
print(train_df['label'].value_counts(normalize=True).round(3))

print("\nLabel distribution in test set:")
print(test_df['label'].value_counts(normalize=True).round(3))

In [ ]:
### Transform into BoW / One-Hot (Fit on Training Data Only)

# 1. Initialize and fit the binarizer ONLY on the training data's entities
mlb = MultiLabelBinarizer()
mlb.fit(train_df["entities"])

print(f"Vocabulary size (based on training set): {len(mlb.classes_)}")

# 2. Transform both the training and test sets using the fitted vocabulary
X_train_bow = mlb.transform(train_df["entities"])
X_test_bow = mlb.transform(test_df["entities"])

# 3. Create the final feature DataFrames
X_train = pd.DataFrame(X_train_bow, columns=mlb.classes_, index=train_df.index)
X_test = pd.DataFrame(X_test_bow, columns=mlb.classes_, index=test_df.index)

# 4. Sanitize feature names (remove characters that XGBoost doesn't allow: [, ], <)
# This is important for XGBoost compatibility
def sanitize_feature_names(df):
    """Replace problematic characters in column names."""
    df.columns = df.columns.str.replace('[', '_', regex=False)
    df.columns = df.columns.str.replace(']', '_', regex=False)
    df.columns = df.columns.str.replace('<', '_', regex=False)
    df.columns = df.columns.str.replace('>', '_', regex=False)
    return df

X_train = sanitize_feature_names(X_train)
X_test = sanitize_feature_names(X_test)

print(f"\nSanitized feature names (removed [, ], <, > characters for XGBoost compatibility)")

# 5. Extract the labels for training and testing
y_train = train_df["label"]
y_test = test_df["label"]

# --- Final Checks ---
print("\nTraining features shape:", X_train.shape)
print("Testing features shape :", X_test.shape)

# Sparsity check on the training data
sparsity = 1 - (X_train.sum().sum() / (X_train.shape[0] * X_train.shape[1]))
print(f"Sparsity of training feature matrix: {sparsity:.3f}")

Training & Evaluation

In [ ]:
### Hyperparameter grids (algorithm/sampling set in first config cell)
HYPERPARAM_GRIDS = {
    'random_forest': {
        "n_estimators": [300, 500],
        "max_depth": [20, 30],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "class_weight": ['balanced', 'balanced_subsample', None]
    },
    'logistic_regression': {
        "C": [0.1, 1.0, 10.0],
        "penalty": ['l1', 'l2'],
        "solver": ['liblinear', 'saga'],
        "class_weight": ['balanced', None]
    },
    'xgboost': {
        "n_estimators": [300, 500],
        "max_depth": [5, 7],
        "learning_rate": [0.1, 0.3],
        "subsample": [1.0],
        "scale_pos_weight": [1, 2, 3]
    },
    'svm': {
        "C": [1.0, 10.0],
        "kernel": ['linear', 'rbf'],
        "gamma": ['scale', 'auto'],
        "class_weight": ['balanced', None]
    },
    'knn': {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ['uniform', 'distance'],
        "metric": ['euclidean', 'manhattan']
    }
}

# Print full experiment configuration (DATA, HORIZON from first cell)
print("=" * 50)
print("EXPERIMENT CONFIGURATION")
print("=" * 50)
print(f"Data           : {DATA}")
print(f"Horizon (days) : {HORIZON}")
print(f"Algorithm      : {ALGORITHM}")
print(f"Sampling       : {SAMPLING}")
print(f"CV Folds       : {CV_FOLDS}")
print(f"Scoring Metric : {SCORING_METRIC}")
print(f"Random State   : {RANDOM_STATE}")
print("=" * 50)

In [ ]:
### (Conditional) Resampling the Training Data

# Only perform SMOTE resampling if the sampling strategy is set to 'smote'
if SAMPLING == 'smote':
    print("=" * 50)
    print("APPLYING SMOTE RESAMPLING")
    print("=" * 50)
    print("Original training set shape %s" % Counter(y_train))
    
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    print("Resampled training set shape %s" % Counter(y_train_resampled))
    print("=" * 50)
else:
    print(f"SMOTE resampling skipped (SAMPLING = '{SAMPLING}')")
    print("Using original training data without resampling.")

In [ ]:
##### Training & Cross-Validation

def get_model_and_grid(algorithm, sampling, hyperparam_grids):
    """
    Returns the model instance and parameter grid based on algorithm and sampling strategy.
    """
    base_grid = hyperparam_grids[algorithm].copy()
    
    # Initialize model based on algorithm
    if algorithm == 'random_forest':
        model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
        # Remove class_weight from grid if not using class_weights sampling
        if sampling != 'class_weights':
            base_grid.pop('class_weight', None)
            
    elif algorithm == 'logistic_regression':
        model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
        # Remove class_weight from grid if not using class_weights sampling
        if sampling != 'class_weights':
            base_grid.pop('class_weight', None)
        # Handle l1 penalty - requires specific solver
        if 'penalty' in base_grid and 'l1' in base_grid['penalty']:
            # Ensure solver supports l1
            if 'solver' in base_grid:
                base_grid['solver'] = ['liblinear', 'saga']
                
    elif algorithm == 'xgboost':
        model = xgb.XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
        # Remove scale_pos_weight if using class_weights or smote
        if sampling in ['class_weights', 'smote']:
            base_grid.pop('scale_pos_weight', None)
            
    elif algorithm == 'svm':
        model = SVC(random_state=RANDOM_STATE, probability=True)  # probability=True for predict_proba
        # Remove class_weight from grid if not using class_weights sampling
        if sampling != 'class_weights':
            base_grid.pop('class_weight', None)
            
    elif algorithm == 'knn':
        model = KNeighborsClassifier(n_jobs=-1)
        # KNN doesn't support class_weight, so remove it if present
        base_grid.pop('class_weight', None)
        
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    return model, base_grid

# Get model and parameter grid
base_model, param_grid = get_model_and_grid(ALGORITHM, SAMPLING, HYPERPARAM_GRIDS)

# Configure data based on sampling strategy
if SAMPLING == 'class_weights':
    print(f"--- Running {ALGORITHM.upper()} with CLASS WEIGHTS strategy ---")
    X_fit, y_fit = X_train, y_train
    
elif SAMPLING == 'smote':
    print(f"--- Running {ALGORITHM.upper()} with SMOTE resampled data ---")
    try:
        X_fit, y_fit = X_train_resampled, y_train_resampled
    except NameError:
        raise NameError("The variables 'X_train_resampled' and 'y_train_resampled' were not found. Please run the SMOTE cell first.")
        
else:  # 'none'
    print(f"--- Running {ALGORITHM.upper()} with NO SAMPLING strategy ---")
    X_fit, y_fit = X_train, y_train

# Setup for CV and GridSearchCV
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Calculate total number of fits for progress tracking
param_combinations = list(product(*[v if isinstance(v, list) else [v] for v in param_grid.values()]))
total_fits = len(param_combinations) * CV_FOLDS

print(f"\n{'='*50}")
print("GRID SEARCH CONFIGURATION")
print(f"{'='*50}")
print(f"Algorithm: {ALGORITHM}")
print(f"Sampling: {SAMPLING}")
print(f"Data shape: {X_fit.shape}")
print(f"Parameter combinations: {len(param_combinations)}")
print(f"CV folds: {CV_FOLDS}")
print(f"Total model fits: {total_fits}")
print(f"{'='*50}")

# Estimate time per fit (quick test)
print("\nEstimating time per fit...")
test_model = base_model.__class__(**{k: v[0] if isinstance(v, list) else v 
                                      for k, v in param_grid.items() if k != 'class_weight'})
test_start = time.time()
test_model.fit(X_fit[:min(100, len(X_fit))], y_fit[:min(100, len(y_fit))])
test_time = time.time() - test_start
estimated_time_per_fit = test_time * (len(X_fit) / min(100, len(X_fit)))

print(f"Estimated time per fit: {estimated_time_per_fit:.2f} seconds")
print(f"Estimated total time: {(estimated_time_per_fit * total_fits)/60:.1f} minutes")

# Create progress bar
pbar = tqdm(total=total_fits, desc="Grid Search", unit="fit",
           bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
           dynamic_ncols=True)

# Track progress
start_time = time.time()
last_update_time = start_time
completed_fits = 0

# Function to update progress bar based on time
def update_progress():
    global completed_fits, last_update_time
    while completed_fits < total_fits:
        elapsed = time.time() - start_time
        # Estimate completed based on time (rough estimate)
        if estimated_time_per_fit > 0:
            estimated_completed = min(int(elapsed / estimated_time_per_fit), total_fits)
            if estimated_completed > completed_fits:
                pbar.update(estimated_completed - completed_fits)
                completed_fits = estimated_completed
                # Update description with current info
                rate = completed_fits / elapsed if elapsed > 0 else 0
                remaining = (total_fits - completed_fits) / rate if rate > 0 else 0
                pbar.set_postfix({
                    'elapsed': f"{elapsed/60:.1f}m",
                    'est. remaining': f"{remaining/60:.1f}m" if remaining > 0 else "?",
                    'rate': f"{rate:.1f} fits/s"
                })
        time.sleep(1)  # Update every second

# Start progress updater in background thread
progress_thread = threading.Thread(target=update_progress, daemon=True)
progress_thread.start()

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring=SCORING_METRIC,
    cv=cv,
    n_jobs=-1,
    verbose=1  # Set to 1 for less verbose output, or 2 for more detail
)

print(f"\nStarting grid search...")
print(f"Progress bar updates every second based on time estimates.\n")

# Run grid search
try:
    grid_search.fit(X_fit, y_fit)
finally:
    # Ensure progress bar shows 100% when done
    pbar.n = total_fits
    pbar.refresh()
    pbar.close()

elapsed_time = time.time() - start_time

print(f"\n{'='*50}")
print(f"Data: {DATA} | Horizon: {HORIZON}")
print(f"Algorithm: {ALGORITHM}")
print("GRID SEARCH COMPLETE")
print(f"{'='*50}")
print(f"Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
print(f"Average time per fit: {elapsed_time/total_fits:.2f} seconds")
print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best CV {SCORING_METRIC.upper()}: {grid_search.best_score_:.3f}")
print(f"{'='*50}")

In [ ]:
# Train final model on full training set using best hyperparameters
best_model = grid_search.best_estimator_

# Retrain on original training data (not resampled) for final evaluation
# This ensures the model is evaluated on real-world data distribution
print(f"\nRetraining best {ALGORITHM} model on original training data...")
best_model.fit(X_train, y_train)
print("Model training complete!")

In [ ]:
##### Learning Curve

# Compute learning curve data
print(f"Computing learning curve for {ALGORITHM}...")
train_sizes, train_scores, val_scores = learning_curve(
    best_model,
    X_train,
    y_train,
    cv=CV_FOLDS,
    scoring=SCORING_METRIC,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    shuffle=True,
    random_state=RANDOM_STATE
)

In [ ]:
# Calculate means and stds
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, "o-", color="tab:blue", label=f"Training {SCORING_METRIC.upper()}")
plt.plot(train_sizes, val_mean, "o-", color="tab:orange", label=f"Validation {SCORING_METRIC.upper()}")

# Confidence bands
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="tab:blue")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="tab:orange")

plt.title(f"Learning Curve ({ALGORITHM.replace('_', ' ').title()})")
plt.xlabel("Training Set Size")
plt.ylabel(f"{SCORING_METRIC.upper()} Score")
plt.legend(loc="best")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
### Model Evaluation

##### Metrics and Confusion Matrix

# Evaluate on test set
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Core metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n" + "=" * 50)
print(f"TEST SET PERFORMANCE ({ALGORITHM.upper()})")
print("=" * 50)
print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1-score : {f1:.3f}")
print(f"ROC-AUC  : {roc_auc:.3f}")
print("=" * 50)

# Classification report for full breakdown by class
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, digits=3))

# Confusion matrix (formatted)
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Control (0)', 'Treatment (1)'], 
            yticklabels=['Control (0)', 'Treatment (1)'])
plt.title(f'Confusion Matrix – {ALGORITHM.replace("_", " ").title()} (Default Threshold)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
### Threshold Tuning

# Get predicted probabilities for the positive class (label 1)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Calculate precision, recall, and thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Find the threshold that maximizes the F1 score
# Note: We calculate F1 for all thresholds except the last one
f1_scores = (2 * precisions * recalls) / (precisions + recalls)
best_f1_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

print(f"Best Threshold (maximizes F1 score): {best_threshold:.4f}")
print(f"F1 Score at this threshold: {best_f1:.4f}")

# --- Plot the Precision-Recall Curve ---
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
plt.plot(thresholds, f1_scores[:-1], "r-", label="F1 Score", linewidth=2)
plt.axvline(x=best_threshold, color='k', linestyle='--', label=f'Best Threshold ({best_threshold:.2f})')
plt.title(f'Precision, Recall, and F1 Score by Decision Threshold ({ALGORITHM.replace("_", " ").title()})')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate Model with the New Threshold
print("\n" + "=" * 50)
print(f"PERFORMANCE WITH OPTIMAL THRESHOLD ({ALGORITHM.upper()})")
print("=" * 50)
y_pred_new_threshold = (y_prob >= best_threshold).astype(int)

# Core metrics
new_acc = accuracy_score(y_test, y_pred_new_threshold)
new_prec = precision_score(y_test, y_pred_new_threshold)
new_rec = recall_score(y_test, y_pred_new_threshold)
new_f1 = f1_score(y_test, y_pred_new_threshold)

print(f"Threshold used: {best_threshold:.4f}")
print(f"Accuracy : {new_acc:.3f}")
print(f"Precision: {new_prec:.3f}")
print(f"Recall   : {new_rec:.3f}")
print(f"F1-score : {new_f1:.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f} (Unaffected by threshold)")
print("=" * 50)

# Plot new confusion matrix
cm_new = confusion_matrix(y_test, y_pred_new_threshold)
print("\nNew Confusion Matrix:")
print(cm_new)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_new, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Control (0)', 'Treatment (1)'], 
            yticklabels=['Control (0)', 'Treatment (1)'])
plt.title(f'Confusion Matrix – {ALGORITHM.replace("_", " ").title()} (Optimal Threshold)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
##### Feature Importances

# Feature importances are only available for tree-based models
if ALGORITHM in ['random_forest', 'xgboost']:
    try:
        # Get feature importances
        importances = pd.Series(
            best_model.feature_importances_, 
            index=X_train.columns
        ).sort_values(ascending=False)
        
        top_n = 20
        top_features = importances.head(top_n)
        
        print(f"\nTop {top_n} most predictive entities ({ALGORITHM.replace('_', ' ').title()}):")
        display(top_features)
        
        # Plot feature importances
        plt.figure(figsize=(10, 6))
        sns.barplot(x=top_features.values, y=top_features.index)
        plt.title(f"Top {top_n} Most Predictive Entities ({ALGORITHM.replace('_', ' ').title()})", fontsize=14)
        plt.xlabel("Feature Importance")
        plt.ylabel("Entity")
        plt.tight_layout()
        plt.show()
        
    except AttributeError:
        print(f"Feature importances not available for {ALGORITHM}")
        
elif ALGORITHM == 'logistic_regression':
    # For logistic regression, we can show coefficient magnitudes
    try:
        coef = best_model.coef_[0]
        feature_importance = pd.Series(
            np.abs(coef),
            index=X_train.columns
        ).sort_values(ascending=False)
        
        top_n = 20
        top_features = feature_importance.head(top_n)
        
        print(f"\nTop {top_n} features by absolute coefficient magnitude (Logistic Regression):")
        display(top_features)
        
        # Plot
        plt.figure(figsize=(10, 6))
        sns.barplot(x=top_features.values, y=top_features.index)
        plt.title(f"Top {top_n} Features by Coefficient Magnitude", fontsize=14)
        plt.xlabel("Absolute Coefficient Value")
        plt.ylabel("Entity")
        plt.tight_layout()
        plt.show()
        
    except AttributeError:
        print("Coefficients not available for this model")
        
else:
    print(f"Feature importance analysis not implemented for {ALGORITHM}")
    print("This feature is typically only available for tree-based models and linear models.")

In [ ]:
### Save Experiment Results to CSV

# Calculate class distributions
# Training set (what the model actually saw)
train_class_counts = y_fit.value_counts().sort_index()
train_class_0 = train_class_counts.get(0, 0)
train_class_1 = train_class_counts.get(1, 0)
train_total = len(y_fit)
train_class_balance_ratio = round(train_class_0 / train_class_1, 4) if train_class_1 > 0 else None
train_class_0_pct = round(train_class_0 / train_total * 100, 2) if train_total > 0 else 0
train_class_1_pct = round(train_class_1 / train_total * 100, 2) if train_total > 0 else 0

# Test set
test_class_counts = y_test.value_counts().sort_index()
test_class_0 = test_class_counts.get(0, 0)
test_class_1 = test_class_counts.get(1, 0)
test_total = len(y_test)
test_class_balance_ratio = round(test_class_0 / test_class_1, 4) if test_class_1 > 0 else None
test_class_0_pct = round(test_class_0 / test_total * 100, 2) if test_total > 0 else 0
test_class_1_pct = round(test_class_1 / test_total * 100, 2) if test_total > 0 else 0

# Collect all experiment information
experiment_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data': DATA,
    'horizon_days': HORIZON,
    'algorithm': ALGORITHM,
    'sampling_strategy': SAMPLING,

    # Best hyperparameters (convert dict to string for CSV)
    'best_params': str(grid_search.best_params_),

    # Cross-validation results
    'best_cv_roc_auc': round(grid_search.best_score_, 4),

    # Test set metrics with DEFAULT threshold (0.5)
    'test_accuracy_default': round(acc, 4),
    'test_precision_default': round(prec, 4),
    'test_recall_default': round(rec, 4),
    'test_f1_default': round(f1, 4),
    'test_roc_auc': round(roc_auc, 4),

    # Confusion matrix with default threshold (as string)
    'confusion_matrix_default': str(cm.tolist()),

    # Threshold tuning results
    'optimal_threshold': round(best_threshold, 4),
    'f1_at_optimal_threshold': round(best_f1, 4),

    # Test set metrics with OPTIMAL threshold
    'test_accuracy_optimal': round(new_acc, 4),
    'test_precision_optimal': round(new_prec, 4),
    'test_recall_optimal': round(new_rec, 4),
    'test_f1_optimal': round(new_f1, 4),

    # Confusion matrix with optimal threshold (as string)
    'confusion_matrix_optimal': str(cm_new.tolist()),

    # Training set class distribution
    'train_total': train_total,
    'train_class_0': train_class_0,
    'train_class_1': train_class_1,
    'train_class_0_pct': train_class_0_pct,
    'train_class_1_pct': train_class_1_pct,
    'train_class_balance_ratio': train_class_balance_ratio,

    # Test set class distribution
    'test_total': test_total,
    'test_class_0': test_class_0,
    'test_class_1': test_class_1,
    'test_class_0_pct': test_class_0_pct,
    'test_class_1_pct': test_class_1_pct,
    'test_class_balance_ratio': test_class_balance_ratio,

    # Data info
    'n_train': len(X_train),  # Original training set size (before resampling)
    'n_test': len(X_test),
    'n_features': X_train.shape[1],
    'cv_folds': CV_FOLDS,
    'scoring_metric': SCORING_METRIC,
}

# Define CSV column headers (in order)
csv_headers = [
    'timestamp',
    'data',
    'horizon_days',
    'algorithm',
    'sampling_strategy',
    'best_params',
    'best_cv_roc_auc',
    'test_accuracy_default',
    'test_precision_default',
    'test_recall_default',
    'test_f1_default',
    'test_roc_auc',
    'confusion_matrix_default',
    'optimal_threshold',
    'f1_at_optimal_threshold',
    'test_accuracy_optimal',
    'test_precision_optimal',
    'test_recall_optimal',
    'test_f1_optimal',
    'confusion_matrix_optimal',
    'train_total',
    'train_class_0',
    'train_class_1',
    'train_class_0_pct',
    'train_class_1_pct',
    'train_class_balance_ratio',
    'test_total',
    'test_class_0',
    'test_class_1',
    'test_class_0_pct',
    'test_class_1_pct',
    'test_class_balance_ratio',
    'n_train',
    'n_test',
    'n_features',
    'cv_folds',
    'scoring_metric',
]

# Check if file exists to determine if we need to write headers
file_exists = os.path.exists(RESULTS_FILE)

# Append row to CSV
with open(RESULTS_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=csv_headers)

    # Write header if file is new
    if not file_exists:
        writer.writeheader()
        print(f"Created new results file: {RESULTS_FILE}")

    # Write experiment data
    writer.writerow(experiment_data)
    print(f"\n{'='*50}")
    print("EXPERIMENT RESULTS SAVED")
    print(f"{'='*50}")
    print(f"File: {RESULTS_FILE}")
    print(f"Data: {DATA}")
    print(f"Horizon (days): {HORIZON}")
    print(f"Algorithm: {ALGORITHM}")
    print(f"Sampling: {SAMPLING}")
    print(f"\nTraining Set Class Distribution:")
    print(f"  Class 0: {train_class_0} ({train_class_0_pct}%)")
    print(f"  Class 1: {train_class_1} ({train_class_1_pct}%)")
    print(f"  Balance Ratio (0:1): {train_class_balance_ratio}")
    print(f"\nTest Set Class Distribution:")
    print(f"  Class 0: {test_class_0} ({test_class_0_pct}%)")
    print(f"  Class 1: {test_class_1} ({test_class_1_pct}%)")
    print(f"  Balance Ratio (0:1): {test_class_balance_ratio}")
    print(f"\nBest CV ROC-AUC: {grid_search.best_score_:.4f}")
    print(f"Test F1 (optimal threshold): {new_f1:.4f}")
    print(f"{'='*50}\n")

In [ ]:
# View results
results_df = pd.read_csv(RESULTS_FILE)
results_df.head()

In [ ]:
# Optional: filter to current DATA and HORIZON
mask = (results_df["data"] == DATA)
if HORIZON is None:
    mask &= results_df["horizon_days"].isna()
else:
    mask &= (results_df["horizon_days"] == HORIZON)

display(
    results_df[mask]
    .sort_values("test_f1_optimal", ascending=False)
)

Inference

In [ ]:
# Inference helpers for trained BoW model (best_model + mlb + X_train expected in memory)

import json
from pathlib import Path
import numpy as np
import pandas as pd

# Safety checks
required_vars = ["best_model", "mlb", "X_train"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(
        f"Missing variables: {missing_vars}. "
        "Run training cells first so best_model/mlb/X_train exist."
    )

def _extract_entities_from_timeline_file(timeline_file: Path):
    """Return (patient_id, entities_list) from one timeline JSON file.

    IMPORTANT: must match training normalization in utils.load_and_extract_entities
    (lowercase + strip), otherwise MultiLabelBinarizer silently drops unknown labels
    and every inference row collapses to zeros -> identical predictions.
    """
    with open(timeline_file, "r", encoding="utf-8") as f:
        timeline = json.load(f)

    patient_id = str(timeline.get("patient_id", timeline_file.stem))
    events = timeline.get("events", [])
    entities = [
        e["entity_preferred_name"].lower().strip()
        for e in events
        if e.get("entity_preferred_name")
    ]
    return patient_id, entities

def _expand_input_paths(paths):
    """
    Accept:
      - single file path
      - directory path (loads *.json)
      - list/tuple of file and/or directory paths
    """
    if isinstance(paths, (str, Path)):
        paths = [paths]

    all_files = []
    for p in paths:
        p = Path(p)
        if p.is_dir():
            all_files.extend(sorted(p.glob("*.json")))
        elif p.is_file():
            all_files.append(p)
        else:
            raise FileNotFoundError(f"Path does not exist: {p}")

    if not all_files:
        raise ValueError("No timeline JSON files found in provided path(s).")
    return all_files

def _build_inference_matrix(entity_lists):
    """Transform entity lists to model-ready feature matrix aligned to X_train columns."""
    vocab = set(mlb.classes_)
    total = sum(len(x) for x in entity_lists) or 1
    matched = sum(1 for x in entity_lists for e in x if e in vocab)
    match_pct = 100.0 * matched / total
    print(
        f"Inference vocab match: {matched}/{total} entity mentions "
        f"({match_pct:.2f}%) matched training vocabulary."
    )
    if match_pct < 5.0:
        print(
            "WARNING: very low vocab match. Likely normalization mismatch "
            "(e.g. case/whitespace) between training and inference extractors. "
            "Predictions will be near-constant."
        )

    X_bow = mlb.transform(entity_lists)
    X_inf = pd.DataFrame(X_bow, columns=mlb.classes_)

    if "sanitize_feature_names" in globals():
        X_inf = sanitize_feature_names(X_inf)

    X_inf = X_inf.reindex(columns=X_train.columns, fill_value=0)

    zero_rows = int((X_inf.sum(axis=1) == 0).sum())
    if zero_rows:
        print(f"NOTE: {zero_rows}/{len(X_inf)} inference rows are all-zero.")

    return X_inf

def _resolve_threshold(passed):
    """
    Resolve the inference threshold explicitly. No silent fallback to 0.5.

    Priority:
      1. `passed` argument (if not None)
      2. `INFERENCE_THRESHOLD` global (explicit override set in the notebook)
      3. `best_threshold` global (from the Threshold Tuning cell)
    Raises RuntimeError if none are available.
    """
    g = globals()
    if passed is not None:
        return float(passed), "argument"
    if "INFERENCE_THRESHOLD" in g and g["INFERENCE_THRESHOLD"] is not None:
        return float(g["INFERENCE_THRESHOLD"]), "INFERENCE_THRESHOLD"
    if "best_threshold" in g and g["best_threshold"] is not None:
        return float(g["best_threshold"]), "best_threshold (Threshold Tuning cell)"
    raise RuntimeError(
        "No inference threshold available. Fix one of:\n"
        "  - pass threshold=... to predict_timelines(...)\n"
        "  - set INFERENCE_THRESHOLD = <float> in a cell\n"
        "  - run the 'Threshold Tuning' cell so `best_threshold` is defined"
    )


def predict_timelines(paths, threshold=None, verbose=True):
    """
    Run inference on one or many timeline JSON files.
    Returns a dataframe with probabilities + class predictions.
    """
    files = _expand_input_paths(paths)

    patient_ids = []
    entity_lists = []
    n_entities = []

    for f in files:
        pid, entities = _extract_entities_from_timeline_file(f)
        patient_ids.append(pid)
        entity_lists.append(entities)
        n_entities.append(len(entities))

    X_inf = _build_inference_matrix(entity_lists)

    if hasattr(best_model, "predict_proba"):
        y_prob = best_model.predict_proba(X_inf)[:, 1]
    elif hasattr(best_model, "decision_function"):
        logits = best_model.decision_function(X_inf)
        y_prob = 1 / (1 + np.exp(-logits))
    else:
        raise AttributeError("Model has neither predict_proba nor decision_function.")

    threshold_value, threshold_source = _resolve_threshold(threshold)
    if verbose:
        print(f"Using threshold={threshold_value:.4f} (source: {threshold_source})")

    y_pred = (y_prob >= threshold_value).astype(int)

    results = pd.DataFrame({
        "patient_id": patient_ids,
        "num_entities": n_entities,
        "prob_treatment_class_1": y_prob,
        "pred_label": y_pred,
        "threshold_used": threshold_value,
    }).sort_values("prob_treatment_class_1", ascending=False).reset_index(drop=True)

    return results

In [ ]:
# Run on entire directory
inference_dir = Path("timelines/ed_timelines_clean_hfirstcontact/")
pred_dir = predict_timelines(inference_dir)
print(f"Predictions generated: {len(pred_dir)}")
display(pred_dir.head(20))

In [ ]:
# Save inference outputs
out_csv = Path("nph_bow_ed_inference_predictions.csv")
pred_dir.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")

In [ ]:
# --- Inference diagnostics on pred_dir ---
# Expects: pred_dir from predict_timelines(...), best_model, X_train, mlb (already in notebook)

import numpy as np
import pandas as pd

if "pred_dir" not in globals():
    raise NameError("pred_dir not found. Run inference first (e.g., pred_dir = predict_timelines(...)).")

print("=" * 70)
print("INFERENCE PREDICTION SUMMARY")
print("=" * 70)

n = len(pred_dir)
print(f"Total inferred timelines: {n}")

# Class counts / proportions
pred_counts = pred_dir["pred_label"].value_counts().sort_index()
pred_props = pred_dir["pred_label"].value_counts(normalize=True).sort_index()

n_control = int(pred_counts.get(0, 0))
n_treat = int(pred_counts.get(1, 0))
p_control = float(pred_props.get(0, 0.0))
p_treat = float(pred_props.get(1, 0.0))

print("\nPredicted class distribution:")
print(f"  Pred 0 (control)   : {n_control:>6}  ({p_control:>7.2%})")
print(f"  Pred 1 (treatment) : {n_treat:>6}  ({p_treat:>7.2%})")

# Probability summary
if "prob_treatment_class_1" in pred_dir.columns:
    probs = pred_dir["prob_treatment_class_1"].astype(float)
    print("\nProbability summary (prob_treatment_class_1):")
    print(probs.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    
    print("\nProbability buckets:")
    bins = [-np.inf, 0.2, 0.4, 0.6, 0.8, np.inf]
    labels = ["<=0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", ">0.8"]
    bucketed = pd.cut(probs, bins=bins, labels=labels)
    print(bucketed.value_counts().sort_index())

# Show top confident examples each way
if "prob_treatment_class_1" in pred_dir.columns:
    print("\nMost confident predicted treatment (top 10):")
    display(pred_dir.sort_values("prob_treatment_class_1", ascending=False).head(10))

    print("\nMost confident predicted control (top 10):")
    display(pred_dir.sort_values("prob_treatment_class_1", ascending=True).head(10))

print("\n" + "=" * 70)
print("GLOBAL FEATURE SIGNALS (if available)")
print("=" * 70)

# Easy global feature importance:
# - LogisticRegression / linear SVM: coefficients
# - tree models (RF/XGB): feature_importances_
# - KNN: not available
if hasattr(best_model, "coef_"):
    # Linear model case (e.g., logistic regression)
    coef = best_model.coef_[0]
    feat_imp = pd.Series(coef, index=X_train.columns).sort_values(ascending=False)

    print("Using linear coefficients (global direction):")
    print("\nTop +20 features (push toward treatment / class 1):")
    display(feat_imp.head(20).to_frame("coef"))

    print("\nTop -20 features (push toward control / class 0):")
    display(feat_imp.tail(20).to_frame("coef"))

elif hasattr(best_model, "feature_importances_"):
    # Tree model case
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    print("Using model.feature_importances_ (magnitude only):")
    display(imp.head(30).to_frame("importance"))

else:
    print("Feature importances not easily available for this model type.")

print("\nDone.")